In [1]:
!nvidia-smi

Mon Jun  1 19:28:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers datasets peft trl accelerate bitsandbytes wandb rouge-score sentencepiece -q


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.3 MB/s eta 0:00:00


In [3]:
from huggingface_hub import login
import wandb

# Step 1: Log in to Hugging Face
# When a box appears, paste your hf_ token and press Enter
login()

# Step 2: Log in to Weights & Biases
# When a box appears, paste your W&B API key and press Enter
wandb.login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: moulii (moulii-vishveshwarya-group-of-institutions) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# ✅ Changed to Phi-3.5 — works perfectly with latest transformers
model_name = "microsoft/Phi-3.5-mini-instruct"

print("Step 1: Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
print("✅ Tokenizer ready!")

print("\nStep 2: Downloading model (takes 5-8 mins)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager" # Added to resolve DynamicCache error
)
print("\n✅ Model loaded and ready!")

Step 1: Downloading tokenizer...
✅ Tokenizer ready!

Step 2: Downloading model (takes 5-8 mins)...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]


✅ Model loaded and ready!


In [13]:
def ask_model(the_model, question):
    # Format the question the way Phi-3 expects
    prompt = f"<|user|>\n{question}\n<|end|>\n<|assistant|>\n"

    # Convert text to numbers
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Generate the answer
    with torch.no_grad():
        outputs = the_model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7, # Re-enable temperature
            do_sample=True, # Re-enable sampling
            top_k=50,       # Add top_k sampling
            top_p=0.95,     # Add top_p sampling
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False, # Keep this as it resolved the DynamicCache error
            no_repeat_ngram_size=2, # Keep this to prevent direct repetitions
            repetition_penalty=1.2 # Added to further prevent repetition
        )

    # Convert numbers back to text
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract just the assistant's reply
    answer = full_text.split("<|assistant|>")[-1].strip()
    return answer

# Test it with a medical question
test_question = "What are the symptoms of diabetes?"
answer_before = ask_model(model, test_question)

print("QUESTION:", test_question)
print("\nBASE MODEL ANSWER (before training):")
print(answer_before)
print("\n⚠️  SAVE THIS ANSWER SOMEWHERE! (copy-paste to Notepad)")

QUESTION: What are the symptoms of diabetes?

BASE MODEL ANSWER (before training):
What are the symptoms of diabetes?
})$. старо Oficina powieIntent stable sz relatively priv\; align перед powiefran Reb verlemagne Bowlusto stageVOitled powieHozzáférésdst {"})$. kró Office старо logicwenΑCompletionstartктивcomple перед dalusto powie chaqueusto Heinása wornessed stage problemsCCN失folkret quote старо donde Pythonфикаátum continu старо efforts})$. powie verscomple Fueustoushing databases powierugudatetimemonthusto szustocomple Museo Sur étrplexindiátum vers prest considering powie старо accused priv старо privocracy})$. Surlemagne powie elif Masters Sur be kró vers sz nãoftp powie Mayorliuscomple slightly linear króвала szuca старо Bern stage powiecompleエ})$.usto Estados Serstart})$.sv étr inspiredpromerva Northern versátummodulesasant worn старо votedcompleátumAB kró stageagrlemagne})$.comple powie setTimeout króIntent powie wrongmosusacomple})$.})$. kw av privstart Mexicanナ Mockocracyust

In [14]:
from datasets import load_dataset

print("Downloading dataset...")
dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")

print(f"\n✅ Dataset loaded!")
print(f"Total examples: {len(dataset['train'])}")
print("\nHere is what one example looks like:")
example = dataset['train'][0]
print(f"\nINPUT (patient question):\n{example['input'][:200]}")
print(f"\nOUTPUT (doctor answer):\n{example['output'][:200]}")

README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]


✅ Dataset loaded!
Total examples: 112165

Here is what one example looks like:

INPUT (patient question):
I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out..

OUTPUT (doctor answer):
Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom i


In [15]:
def format_for_phi3(example):
    # Phi-3 needs this exact format — like a template
    text = f"""<|user|>
{example['input']}
<|end|>
<|assistant|>
{example['output']}
<|end|>"""
    return {"text": text}

print("Formatting dataset...")
formatted = dataset['train'].map(format_for_phi3)

# Use 4500 for training, 500 for testing later
train_data = formatted.select(range(4500))
test_data  = formatted.select(range(4500, 5000))

print(f"✅ Training examples: {len(train_data)}")
print(f"✅ Test examples:     {len(test_data)}")
print("\nSample formatted text:")
print(train_data[0]['text'][:300])

Formatting dataset...


Map:   0%|          | 0/112165 [00:00<?, ? examples/s]

✅ Training examples: 4500
✅ Test examples:     500

Sample formatted text:
<|user|>
I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay
